This version uses GPT 2 model as a source model and just wikipedia to find hallocations with threshold 0.8

Install dependencies

In [1]:
!pip install -q transformers torch wikipedia sentence-transformers nltk pandas

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 86.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 11.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.5 MB/s eta 0:00:00:00:0100:01


Imports, config, and setup

In [2]:
import os, re, math
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt', quiet=True)

import wikipedia
from wikipedia.exceptions import DisambiguationError, PageError
wikipedia.set_lang("en")  # later you can switch to "ur" for Urdu

THRESHOLD = 0.8  # your required decision threshold (average similarity)
PER_CLAIM_INFO = True  # keep detailed per-claim table

print("Setup complete. Threshold =", THRESHOLD)


Setup complete. Threshold = 0.8


Load models (GPT-2 generator + sentence embeddings)

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer, util

# ---- LLM: GPT-2 (free) ----
llm_name = "gpt2"
tok = AutoTokenizer.from_pretrained(llm_name)
model = AutoModelForCausalLM.from_pretrained(llm_name)

# GPT-2 has no pad token -> set to eos to avoid warnings
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

device = 0 if torch.cuda.is_available() else -1
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tok,
    device=device
)

# ---- Embeddings for similarity ----
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")

print("LLM:", llm_name, "| Device:", "GPU" if device == 0 else "CPU")


2025-08-22 13:54:09.559330: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755870849.795074      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755870849.864461      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LLM: gpt2 | Device: CPU


Generate a draft answer from GPT-2

In [4]:
def get_draft_answer(user_question: str) -> str:
    """
    Simple Q/A prompt for GPT-2.
    GPT-2 is not instruction-tuned, so we use a QA-styled prefix.
    """
    prompt = f"Q: {user_question}\nA:"
    out = generator(
        prompt,
        max_new_tokens=160,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tok.eos_token_id
    )[0]["generated_text"]

    # Remove the prompt prefix if GPT-2 echoes it
    answer = out[len(prompt):].strip()
    # Basic cleanup (stop at next 'Q:' if model rambles)
    cut = answer.split("\nQ:")[0].strip()
    return cut if cut else answer

print("Draft generator ready.")


Draft generator ready.


Extract factual claims (simple, robust heuristic)

In [5]:
DIGIT_RE = re.compile(r"\d")
PROPER_RE = re.compile(r"\b([A-Z][a-zA-Z]+(?:\s[A-Z][a-zA-Z]+)*)\b")

def extract_claims(text: str):
    """
    Split into sentences and keep those that *look* factual:
    - has a number, or
    - has capitalized tokens (proper-noun-ish), and
    - has at least ~6 words.
    """
    sentences = [s.strip() for s in sent_tokenize(text) if s.strip()]
    claims = []
    for s in sentences:
        if len(s.split()) >= 6 and (DIGIT_RE.search(s) or PROPER_RE.search(s)):
            claims.append(s)
    # If nothing looks factual, fallback to check the whole text as one claim
    if not claims and text.strip():
        claims = [text.strip()]
    return claims

print("Claim extractor ready.")


Claim extractor ready.


Wikipedia retrieval + similarity scoring

In [6]:
def wiki_best_match(query: str, k_titles: int = 5, summary_sents: int = 3):
    """
    Search Wikipedia for 'query', test top-k pages,
    and return the (best_title, best_url, best_summary, best_score_vs_query).
    If nothing solid found, returns (None, None, None, 0.0).
    """
    try:
        titles = wikipedia.search(query)
    except Exception:
        titles = []

    if not titles:
        return None, None, None, 0.0

    q_emb = embedder.encode([query], convert_to_tensor=True)
    best = (0.0, None, None, None)  # (score, title, url, summary)

    for title in titles[:k_titles]:
        try:
            page = wikipedia.page(title, auto_suggest=False, redirect=True)
            summ = wikipedia.summary(title, sentences=summary_sents, auto_suggest=False, redirect=True)
        except DisambiguationError as e:
            # try a couple of options
            tried_any = False
            for opt in e.options[:3]:
                try:
                    page = wikipedia.page(opt, auto_suggest=False, redirect=True)
                    summ = wikipedia.summary(opt, sentences=summary_sents, auto_suggest=False, redirect=True)
                    tried_any = True
                except Exception:
                    continue
                if not summ or len(summ.split()) < 15:
                    continue
                s_emb = embedder.encode([summ], convert_to_tensor=True)
                score = float(util.cos_sim(q_emb, s_emb)[0][0])
                if score > best[0]:
                    best = (score, page.title, page.url, summ)
            if not tried_any:
                continue
        except PageError:
            continue
        except Exception:
            continue

        if not summ or len(summ.split()) < 15:
            continue

        s_emb = embedder.encode([summ], convert_to_tensor=True)
        score = float(util.cos_sim(q_emb, s_emb)[0][0])
        if score > best[0]:
            best = (score, page.title, page.url, summ)

    return best[1], best[2], best[3], best[0]


def verify_claims_with_wikipedia(claims):
    """
    For each claim: find best Wikipedia summary and compute similarity(claim, summary).
    Returns (DataFrame, average_similarity).
    """
    rows = []
    for claim in claims:
        title, url, snippet, _ = wiki_best_match(claim, k_titles=5, summary_sents=3)
        if title is None:
            score = 0.0
            snippet = ""
            url = None
        else:
            c_emb = embedder.encode([claim], convert_to_tensor=True)
            s_emb = embedder.encode([snippet], convert_to_tensor=True)
            score = float(util.cos_sim(c_emb, s_emb)[0][0])

        rows.append({
            "claim": claim,
            "similarity": round(score, 3),
            "source_title": title if title else "",
            "source_url": url if url else "",
            "snippet": snippet[:300].replace("\n", " ")
        })

    df = pd.DataFrame(rows)
    avg = float(df["similarity"].mean()) if not df.empty else 0.0
    return df, avg

print("Wikipedia verifier ready.")


Wikipedia verifier ready.


End-to-end pipeline (with your threshold rule)

In [7]:
REFUSAL_MESSAGE = "i donot exactly know the answer can you clear your prompt please"

def run_pipeline_interactive():
    question = input("Enter your question: ").strip()
    if not question:
        print("No question provided.")
        return

    print("\n--- Generating draft answer (GPT-2) ---")
    draft = get_draft_answer(question)
    print(draft, "\n")

    print("--- Extracting claims ---")
    claims = extract_claims(draft)
    if not claims:
        print("No claims detected; treating entire draft as one claim.")
        claims = [draft]
    print(f"Detected {len(claims)} claim(s).\n")

    print("--- Verifying against Wikipedia ---")
    df, avg = verify_claims_with_wikipedia(claims)

    if PER_CLAIM_INFO and not df.empty:
        display(df[["claim", "similarity", "source_title", "source_url"]])

    print(f"\nAverage similarity score: {avg:.3f}  |  Decision threshold: {THRESHOLD:.2f}")

    if avg >= THRESHOLD:
        print("\n✅ Threshold met — returning the model's answer:\n")
        print(draft)
    else:
        print("\n❌ Threshold NOT met — refusing with your message:\n")
        print(REFUSAL_MESSAGE)


Run it!

In [13]:
run_pipeline_interactive()

Enter your question:  who is Allah



--- Generating draft answer (GPT-2) ---
Allah is the Most Merciful. He is the One Who has made for us the living things we need and desire. Allah is the Most Merciful. He is the One Who has made for us the living things we need and desire. 

--- Extracting claims ---
Detected 2 claim(s).

--- Verifying against Wikipedia ---


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,claim,similarity,source_title,source_url
0,He is the One Who has made for us the living t...,0.349,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life
1,He is the One Who has made for us the living t...,0.349,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life



Average similarity score: 0.349  |  Decision threshold: 0.80

❌ Threshold NOT met — refusing with your message:

i donot exactly know the answer can you clear your prompt please
